<a href="https://colab.research.google.com/github/hrley55/BIG-DATA-NHOM-7/blob/main/Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import datetime as dt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from google.colab import drive
drive.mount('/content/drive')
# ==========================================
# BƯỚC 1: PANDAS - GỘP FILE VÀ TÍNH RFM
# ==========================================
# 1. Đọc 3 file data
df_customers = pd.read_csv('/content/drive/MyDrive/Olist_Data/olist_customers_dataset.csv')
df_orders = pd.read_csv('/content/drive/MyDrive/Olist_Data/olist_orders_dataset.csv')
df_payments = pd.read_csv('/content/drive/MyDrive/Olist_Data/olist_order_payments_dataset.csv')

# 2. Merge các file lại với nhau
df_merged = df_orders.merge(df_customers, on='customer_id').merge(df_payments, on='order_id')

# Chuyển cột thời gian về định dạng datetime
df_merged['order_purchase_timestamp'] = pd.to_datetime(df_merged['order_purchase_timestamp'])

# 3. Tính toán R, F, M
reference_date = df_merged['order_purchase_timestamp'].max() + dt.timedelta(days=1)

df_rfm = df_merged.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (reference_date - x.max()).days, # Recency
    'order_id': 'nunique',                                                 # Frequency
    'payment_value': 'sum'                                                 # Monetary
}).reset_index()

df_rfm.columns = ['customer_unique_id', 'Recency', 'Frequency', 'Monetary']

# IN BẢNG THỐNG KÊ ĐỂ COPY VÀO BÁO CÁO
print(df_rfm.describe())

# ==========================================
# BƯỚC 2: SKLEARN PIPELINE - CHUẨN HÓA & PHÂN CỤM
# ==========================================
# Tách các cột số để đưa vào Pipeline
X_rfm = df_rfm[['Recency', 'Frequency', 'Monetary']]

# Khởi tạo Pipeline: StandardScaler -> KMeans
rfm_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('kmeans', KMeans(n_clusters=4, random_state=42, n_init=10))
])

# Chạy pipeline và gán nhãn cụm (cluster) cho khách hàng
df_rfm['Cluster'] = rfm_pipeline.fit_predict(X_rfm)

Mounted at /content/drive
            Recency     Frequency      Monetary
count  96095.000000  96095.000000  96095.000000
mean     288.730756      1.034809    166.594226
std      153.407846      0.214385    231.428912
min        1.000000      1.000000      0.000000
25%      164.000000      1.000000     63.120000
50%      269.000000      1.000000    108.000000
75%      398.000000      1.000000    183.530000
max      773.000000     17.000000  13664.080000


In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression # Ví dụ mô hình Classification

# Tải bộ từ dừng (Stop words) tiếng Bồ Đào Nha
nltk.download('stopwords')
pt_stop_words = stopwords.words('portuguese')

# 1. Đọc file
df_reviews = pd.read_csv('/content/drive/MyDrive/Olist_Data/olist_order_reviews_dataset.csv')

# 2. Xử lý Null (thay NaN bằng chuỗi rỗng) - Làm bằng Pandas
df_reviews['review_comment_message'] = df_reviews['review_comment_message'].fillna('')

# Lấy dữ liệu Text (X) và Nhãn dự đoán (y) - ví dụ dự đoán điểm review
X_text = df_reviews['review_comment_message']
y_label = df_reviews['review_score']

# ==========================================
# BƯỚC 3: XÂY DỰNG TEXT PIPELINE
# ==========================================
text_pipeline = Pipeline(steps=[
    ('tfidf', TfidfVectorizer(
        max_features=5000,           # Lấy 5000 từ quan trọng nhất
        min_df=5,                    # Loại bỏ từ xuất hiện < 5 lần (từ hiếm, sai chính tả)
        max_df=0.8,                  # Loại bỏ từ có mặt ở > 80% bình luận (từ quá phổ biến)
        stop_words=pt_stop_words     # Bỏ các từ vô nghĩa tiếng Bồ Đào Nha
    )),
    ('classifier', LogisticRegression(max_iter=1000)) # Mô hình phân loại
])

# 4. Trích xuất kích thước ma trận để báo cáo
# Ta có thể gọi riêng bước tfidf trong pipeline để xem chiều dữ liệu
tfidf_step = text_pipeline.named_steps['tfidf']
sparse_matrix = tfidf_step.fit_transform(X_text)

# IN KÍCH THƯỚC MA TRẬN ĐỂ COPY VÀO BÁO CÁO
print(f"Output sparse matrix dimension: {sparse_matrix.shape}")
# (Sẽ ra một số kiểu như: (99224, 5000))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Output sparse matrix dimension: (99224, 3551)
